# 01 - Data Understanding

**Goal of this notebook:** know exactly what data we have *before* modelling - what a row means, what the target means,
what is missing or odd, and whether any column could leak the answer.

> **Why this stage matters.** Most real-world modelling failures are data problems (wrong target definition, leakage, silent
> missing values), not algorithm problems. A model built on a misunderstood dataset can look excellent and be useless.

## Dataset source and context
* **Dataset:** *Telco Customer Churn* - an IBM sample dataset (IBM Community / Cognos Analytics sample data), downloaded from IBM's public
  GitHub repository `IBM/telco-customer-churn-on-icp4d`. It is also widely mirrored on Kaggle.
* **Business story:** a telecom company wants to understand which customers leave. **One row = one customer.**
* **Important limitation:** this is a *sample* dataset for a fictional telecom. It is perfect for learning and for demonstrating a
  correct workflow, but conclusions about "what causes churn" do **not** transfer to any real company.

## Data dictionary
| Column | Meaning |
|---|---|
| `customerID` | unique identifier (not a feature) |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | demographics / household |
| `tenure` | months the customer has been with the company |
| `PhoneService`, `MultipleLines` | phone products |
| `InternetService` | DSL / Fiber optic / No |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | add-on services (only available with internet) |
| `Contract` | Month-to-month / One year / Two year |
| `PaperlessBilling`, `PaymentMethod` | billing preferences |
| `MonthlyCharges` | current monthly bill |
| `TotalCharges` | total billed to date |
| **`Churn`** | **target**: "Yes" = the customer left (per IBM: left within the last month), "No" = still a customer |

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                       # lets notebooks import the src/ package

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    """Save the current figure into reports/figures (used by README and dashboard)."""
    plt.savefig(config.FIGURES_DIR / name, dpi=150, bbox_inches="tight")

In [2]:
from src.data_preprocessing import load_raw_data, validate_schema, clean_data, summarize_dataset
raw = load_raw_data()
validate_schema(raw)
print("Rows x columns:", raw.shape)
raw.head()

Rows x columns: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Types, missing values, uniqueness
`pandas` reads `TotalCharges` as text. A numeric column stored as text almost always means *something non-numeric is hiding in it*.

In [3]:
profile = pd.DataFrame({"dtype": raw.dtypes.astype(str), "n_unique": raw.nunique(), "n_missing(NaN)": raw.isna().sum()})
profile

,dtype,n_unique,n_missing(NaN)
customerID,str,7043,0
gender,str,2,0
SeniorCitizen,int64,2,0
Partner,str,2,0
Dependents,str,2,0
tenure,int64,73,0
PhoneService,str,2,0
MultipleLines,str,3,0
InternetService,str,3,0
OnlineSecurity,str,3,0


In [4]:
# Investigate the suspicious text column instead of just coercing it blindly.
tc_numeric = pd.to_numeric(raw["TotalCharges"].str.strip(), errors="coerce")
blank = raw[tc_numeric.isna()]
print("Rows where TotalCharges cannot be parsed as a number:", len(blank))
display(blank[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])
print("\nAll of them have tenure == 0:", bool((blank["tenure"] == 0).all()))
print("Churn values among them:", blank["Churn"].value_counts().to_dict())

Rows where TotalCharges cannot be parsed as a number: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No



All of them have tenure == 0: True
Churn values among them: {'No': 11}


**Reading the output.** The blanks are not random: every one is a customer with `tenure == 0` (brand-new, nothing billed yet), and none
of them churned. So the blank really means "no charges yet" = **0**, which is a rule derived from understanding the business, *not* a statistic
(like the median) learned from other customers.

> **Common mistake:** `df.dropna()` or median-imputing here. Dropping removes 11 legitimate new customers; median-imputing would invent
> a bill of ~$1,400 for someone who has not been billed.

Because this rule is a deterministic per-row fix (it uses no information from other rows), applying it before the train/test split is safe.

## 2. Duplicates
Two different questions: *is the same customer listed twice?* (a data error) vs *do two different customers have identical attributes?*
(normal when most columns are categorical).

In [5]:
feature_cols = [c for c in raw.columns if c != "customerID"]
print("Duplicate customerID values:      ", raw["customerID"].duplicated().sum())
print("Rows identical in every feature:   ", raw.duplicated(subset=feature_cols).sum(), "(ignoring customerID)")
dups = raw[raw.duplicated(subset=feature_cols, keep=False)].sort_values(feature_cols)
display(dups.head(6))

Duplicate customerID values:       0
Rows identical in every feature:    22 (ignoring customerID)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
6491,9728-FTTVZ,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.20,69.2,Yes
6764,7660-HDPJV,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.20,69.2,Yes
4495,4702-IOQDC,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.10,70.1,Yes
6267,0328-GRPMV,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.10,70.1,Yes
5522,2619-WFQWU,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes
5759,9985-MWVIX,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes


**Decision:** `customerID` is unique, so there are no repeated customers. The few rows identical in all features have *different* IDs - with 16 mostly-categorical
columns and ~7,000 customers, some coincidences are expected. Removing them is **not justified** (we would be deleting real customers), so we keep them.

## 3. Target definition and class balance

In [6]:
clean = clean_data(raw)
counts = clean["Churn"].value_counts().rename({0: "Retained (0)", 1: "Churned (1)"})
print(counts.to_string()); print(f"\nChurn rate: {clean['Churn'].mean():.2%}")
print(f"A model that always predicts 'stays' would be {1-clean['Churn'].mean():.2%} accurate -> accuracy is a weak metric here.")

Churn
Retained (0)    5174
Churned (1)     1869

Churn rate: 26.54%
A model that always predicts 'stays' would be 73.46% accurate -> accuracy is a weak metric here.


**Class imbalance:** roughly 1 in 4 customers churn. That is *moderate* imbalance - not extreme (like fraud at 0.1%). It is enough that
(a) accuracy becomes misleading and (b) we should prefer precision/recall-type metrics and PR-AUC, but not so severe that special resampling is
automatically needed. We revisit this when modelling.

## 4. Leakage and "available at prediction time" review
*Target leakage* = a feature that contains information from after the outcome (or is a proxy for the label), so the model "cheats" in training but would not have it in real use.

| Column | Verdict | Reasoning |
|---|---|---|
| `customerID` | **Drop** | Pure identifier. If IDs happen to be assigned over time, a model could learn patterns that do not generalise. |
| `tenure`, `TotalCharges` | **Keep, with caution** | Legitimate features *if measured as of the prediction date*. In this snapshot, churned customers' tenure/charges stop at the churn date, so they are partly a function of *when* the customer left. In a real deployment you would compute them at a fixed "scoring date" before the outcome window. |
| Services, contract, billing, demographics | **Keep** | Describe the customer's plan; available before churn. |
| Churn reason / churn date / cancellation score | **Not present** | Columns like these (present in some versions of this dataset) would be leakage - good that we do not have them. |

Let us check one specific relationship: `TotalCharges` is supposed to be roughly `tenure x MonthlyCharges`.

In [7]:
m = clean[clean["tenure"] > 0]
ratio = m["TotalCharges"] / (m["tenure"] * m["MonthlyCharges"])
print("Correlation(TotalCharges, tenure*MonthlyCharges):", round(m["TotalCharges"].corr(m["tenure"] * m["MonthlyCharges"]), 4))
print(ratio.describe().round(3).to_string())

Correlation(TotalCharges, tenure*MonthlyCharges): 0.9996
count    7032.000
mean        1.000
std         0.051
min         0.689
25%         0.980
50%         1.000
75%         1.020
max         1.573


`TotalCharges` is largely redundant with `tenure` and `MonthlyCharges` (very strongly correlated, with deviations explained by price changes/promotions).
Redundant features are not "wrong", but they make *individual* coefficients and importances harder to interpret - something to remember in the SHAP notebook.

**Other caveats to record:** no timestamps (so no out-of-time validation), a single snapshot, fictional company, and attributes like `gender` / `SeniorCitizen`
that raise fairness questions if used to decide who receives retention offers.

## 5. Train / test split (done *before* any exploration that informs modelling)
We hold out 20% of customers as a **test set** that is not used for exploration, feature decisions, model selection, tuning or threshold choice.
All of those use only the 80% *development set*; inside it, cross-validation provides the "validation" folds.
The split is **stratified** on `Churn` (same churn rate in both parts) and **seeded** (`random_state=42`) so it is reproducible.

In [8]:
from src.data_preprocessing import load_split
train, test = load_split("train"), load_split("test")
summary = pd.DataFrame({"customers": [len(train), len(test)], "churn_rate": [train.Churn.mean(), test.Churn.mean()]}, index=["development (train)", "held-out test"])
display(summary.style.format({"churn_rate": "{:.2%}"}))
print("Overlap in customerIDs:", len(set(train.customerID) & set(test.customerID)))
import json; print(json.dumps(json.loads(config.DATA_SUMMARY_PATH.read_text()), indent=2))

,customers,churn_rate
development (train),5634,26.54%
held-out test,1409,26.54%


Overlap in customerIDs: 0
{
  "n_rows": 7043,
  "n_columns": 21,
  "n_unique_customers": 7043,
  "churn_rate": 0.2653698707936959,
  "n_churned": 1869,
  "n_retained": 5174,
  "duplicate_customer_ids": 0,
  "duplicate_rows_ignoring_id": 22,
  "unparsable_total_charges": 11,
  "unparsable_total_charges_all_tenure_zero": true,
  "missing_values_after_cleaning": 0,
  "n_train": 5634,
  "n_test": 1409,
  "train_churn_rate": 0.2653532126375577,
  "test_churn_rate": 0.2654364797728886,
  "test_size": 0.2,
  "random_state": 42
}


> **Limitation of random splitting.** A random split assumes customers are independent and that future customers resemble past ones.
> With real data you would usually split **by time** (train on earlier months, test on later) to detect drift. This dataset has no dates, so we cannot, and our test score is
> probably a somewhat *optimistic* view of future performance.

## What to remember / interview questions
* *"You find a numeric column stored as text. What do you do?"* - Look at the offending values before coercing; check whether they follow a rule (here: new customers).
* *"How do you decide whether to drop duplicates?"* - Distinguish duplicate *entities* (error) from identical *attributes* (can be legitimate).
* *"Where might leakage hide in a churn dataset?"* - Columns created after the outcome (cancellation reason, final-bill adjustments), IDs/timestamps correlated with the label, features computed over a window that extends past the prediction date.